# PufferLib 5.0 native MinGRU on a Colab GPU

Our native trainer (`forecast_train.cu`, PufferLib 5.0's MinGRU + Muon in CUDA C, our supervised loss) built against PufferLib at the pinned commit, `src/` unchanged.

**Where the data comes from.** Nothing is read from the repo or the clusters. Everything comes from one zip on your Google Drive, built on the laptop:

    python experiments/puffer_forecast/colab_bundle.py phase2c

→ `results/puffer_forecast_2026-10-12/_work/colab/puffer_colab_phase2c.zip` (~12 MB). Upload it to `MyDrive/puffer_forecast/`. It holds:
- `data/<design>/X.f32, y.f32, meta.txt`: the 16:00-bar design of record for `baseline` and `all_features` (the regular stack's **prescaled** design, rows × columns, float32), and the target `y` (sqrt fit space);
- `dates.csv`: row → 16:00 session (2010-07-12 … 2024-04-30);
- `native/`: the trainer source; `inputs/<phase>/`: that phase's walk-forward job files; `phase2_select.py`;
- `MANIFEST.md5`: every file's md5, checked after unzipping.

The notebook mounts Drive (Google asks for permission), unzips to `/content/pf`, fetches PufferLib from GitHub at the pinned commit, and builds for whatever GPU Colab gave you. Outputs go to `/content/work` and are copied to `MyDrive/puffer_forecast/`.

**Part A: manual training.** Write job lines (one model each), train them, look at the curves and forecasts, change things, run again. A fit takes about a second.
**Part B: the full walk-forward** (tune → select → refit at every session, as the cluster job). Off unless `RUN_WALK = True`.

In [ ]:
# ---- settings
PHASE = "phase2c"                      # must match the bundle
DRIVE_DIR = "/content/drive/MyDrive/puffer_forecast"
BUNDLE = f"{DRIVE_DIR}/puffer_colab_{PHASE}.zip"
COMMIT = "6ffa5b10dbbbe4d1e8288367c7d9d3acd3bad4a2"   # PufferLib 5.0, as on CARC / Hoffman2
CONCURRENT = 48                        # models trained at once
CHUNK = 5
POOL = True                            # --pool: reuse trainers between jobs of one shape (byte-identical, faster)
LINK = "softplus"                      # Part A's output link: softplus (f > 0 always) or identity (Phases 2-2c)
EXTRA_FLAGS = ""                       # Part A only, e.g. " --tf32-ns" (TF32 in Muon's Newton-Schulz; changes numerics)
RUN_WALK = False                      # Part B: the full walk-forward
RUN_NAME = f"{PHASE}_colab"            # Part B's Drive folder; the same name resumes

In [ ]:
# ---- helpers, GPU check
import datetime, hashlib, json, os, pathlib, shutil, subprocess, time, zipfile

CUDA = "/usr/local/cuda"
os.environ["PATH"] = f"{CUDA}/bin:" + os.environ["PATH"]
os.environ["CUDA_DEVICE_MAX_CONNECTIONS"] = "32"   # 32 hardware queues, not 8: many small models in parallel (gate 4)


def sh(cmd, cwd=None, quiet=False):
    """Run a bash command, streaming its output; raise if it fails. Returns the output."""
    if not quiet:
        print("$", cmd, flush=True)
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = []
    for line in p.stdout:
        if not quiet:
            print(line, end="", flush=True)
        out.append(line)
    if p.wait() != 0:
        if quiet:
            print("".join(out[-40:]))
        raise RuntimeError(f"exit {p.returncode}: {cmd}")
    return "".join(out)


q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                   capture_output=True, text=True)
assert q.returncode == 0 and q.stdout.strip(), "no GPU: Runtime > Change runtime type > GPU"
GPU = q.stdout.strip().splitlines()[0]
CAP = GPU.split(",")[-1].strip().replace(".", "")   # e.g. 75 (T4), 89 (L4), 80 (A100), 90 (H100)
print("GPU:", GPU, "-> sm_" + CAP)
sh("nvcc --version | tail -2; gcc --version | head -1")

In [ ]:
# ---- Drive, bundle (checked against its manifest)
from google.colab import drive
drive.mount("/content/drive")

ROOT = pathlib.Path("/content/pf")
shutil.rmtree(ROOT, ignore_errors=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall(ROOT)
bad = []
for line in (ROOT / "MANIFEST.md5").read_text().splitlines():
    h, rel = line.split("  ", 1)
    if hashlib.md5((ROOT / rel).read_bytes()).hexdigest() != h:
        bad.append(rel)
assert not bad, f"bundle files differ from its manifest: {bad}"
INFO = json.loads((ROOT / "bundle.json").read_text())
print(json.dumps(INFO, indent=1))
assert INFO["phase"] == PHASE, f"bundle is {INFO['phase']}, PHASE is {PHASE}"
assert INFO["puffer_commit"] == COMMIT

In [ ]:
# ---- PufferLib 5.0 at the pinned commit (only that commit is fetched); src/ must be unmodified
PUF = pathlib.Path("/content/PufferLib5")
if not (PUF / ".git").exists():
    sh(f"git init -q {PUF} && git -C {PUF} remote add origin https://github.com/PufferAI/PufferLib.git")
    sh(f"git -C {PUF} fetch -q --depth 1 origin {COMMIT} && git -C {PUF} checkout -q FETCH_HEAD")
assert sh(f"git -C {PUF} rev-parse HEAD").strip() == COMMIT, "PufferLib checkout is not the pinned commit"
sh(f"git -C {PUF} diff --quiet -- src")

In [ ]:
# ---- build (same flags as the cluster jobs, for this GPU only)
BUILD = pathlib.Path("/content/build")
BUILD.mkdir(exist_ok=True)
BIN = BUILD / "forecast_train"
FLAGS = (f"-O2 --threads 4 -Wno-deprecated-gpu-targets -std=c++17 -gencode arch=compute_{CAP},code=sm_{CAP} "
         f"-DPRECISION_FLOAT -I. -I{PUF}/src -I{PUF}/vendor -I{CUDA}/include -Xcompiler=-Wno-narrowing "
         "--diag-suppress=2361 --diag-suppress=111 --diag-suppress=128 --diag-suppress=177 --diag-suppress=550")
sh(f"nvcc {FLAGS} forecast_train.cu -L{CUDA}/lib64 -lcublas -lcurand -o {BIN} > {BUILD}/build.log 2>&1 "
   f"|| (cat {BUILD}/build.log; exit 1)", cwd=ROOT / "native")
POOL_FLAG = " --pool" if POOL and "--pool" in (ROOT / "native" / "forecast_train.cu").read_text() else ""
print("built", BIN, "| trainer pool:", "on" if POOL_FLAG else "off (not in this source)")

## Part A: manual training

**A job line is one model.** Keys (anything left out takes the default in `BASE`, which is Phase 2c's setting):

| key | meaning |
|---|---|
| `data` | `data/baseline` or `data/all_features` |
| `w0`, `w1` | training window: rows `[w0, w1)` (use `window(end_date)`) |
| `mode` | `tune`: fit on `[w0, w1 − val_tail − embargo)`, early-stop on the last `val_tail` rows; `refit`: fit on the whole window for `epochs` epochs and forecast the next `fc_n` rows |
| `L`, `H`, `layers` | sequence length (sessions), hidden width, MinGRU layers |
| `lr`, `momentum`, `max_grad_norm`, `anneal`, `min_lr_ratio` | Muon learning rate and schedule (cosine anneal to `min_lr_ratio`) |
| `batch`, `max_epochs`, `patience`, `epochs` | minibatch size, early-stopping cap / patience (tune), fixed epochs (refit) |
| `loss` | `mse` (fit space), `qlike`, `huber` (`huber_q`, `qlike_eps`, `qlike_cap_q`) |
| `mask` | 1: drop the window's constant / duplicate columns (the regular stack's rule, causal) |
| `xstd` | 0: feed the prescaled design as is (**the fix**); 1: re-standardise inputs on the window (the old bug) |
| `seed` | initial weights and minibatch order |
| `link` | `softplus`: f = softplus(c + s·o), always > 0 (c, s set so the start matches the identity scale); `identity`: f = mu_y + sd_y·o (Phases 2–2c). The loss stays in sqrt space either way |
| `stop_loss` | refit only: stop once the epoch's mean training loss, relative to the window's target variance, is ≤ this (e.g. `1e-3`) |
| `fc_train` | 1: also write the final weights' forecast of every training row (`preds` kind `train`): the in-sample fit |

**Outputs** (`train()` returns them as DataFrames): `jobs` (one row per model: best epoch, best validation MSE, status, timings), `epochs` (validation MSE / QLIKE per epoch), `steps` (training loss and gradient norm per step), `preds` (`val` = the validation tail of a tune job at its best epoch, `fc` = a refit's forecasts; `row` indexes `dates` and `y`).

In [ ]:
# ---- Part A setup: data, defaults, helpers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATES = pd.read_csv(ROOT / "dates.csv", parse_dates=["date"])
Y = {}
for d in ("baseline", "all_features"):
    n, p = map(int, (ROOT / "data" / d / "meta.txt").read_text().split()[:2])
    Y[d] = np.fromfile(ROOT / "data" / d / "y.f32", dtype="<f4")
    print(f"{d}: {n} rows x {p} columns; target y (sqrt fit space) {len(Y[d])} rows")
print("sessions:", DATES.date.iloc[0].date(), "...", DATES.date.iloc[-1].date())

blocks = json.loads((ROOT / "inputs" / PHASE / "blocks.json").read_text())
BASE = dict(blocks["common"], mode="tune", loss="mse", mask=1, xstd=0)
if "softplus" in (ROOT / "native" / "forecast_train.cu").read_text():
    BASE["link"] = LINK
else:
    print("this trainer has no link option: identity output")
BASE.update({k: v for k, v in blocks.items() if k in ("fc_n",)})
print("defaults:", BASE)
W = int(blocks.get("W", 2000))
SEEDS = [int(s) for s in blocks["seeds"]]   # the cluster's seeds (tune and refit use the same ones)
WORK = pathlib.Path("/content/work")


def window(end_date, w=W):
    """(w0, w1): the w sessions ending on end_date (inclusive)."""
    w1 = int((DATES.date <= pd.Timestamp(end_date) + pd.Timedelta(hours=16)).sum())
    assert w1 - w >= 0, "window starts before the first session"
    return w1 - w, w1


def job(**kw):
    """One job line; kw overrides BASE. Unless given, id = mode, design and every setting that differs from BASE."""
    j = dict(BASE, **kw)
    if "id" not in j:
        diff = [f"{k}{v}" for k, v in j.items() if k not in ("mode", "data") and BASE.get(k) != v]
        j["id"] = "_".join([str(j["mode"]), str(j["data"]).split("/")[-1], *diff])
    return " ".join(f"{k}={v}" for k, v in j.items())


def train(lines, name):
    """Train the job lines (one model each) on the GPU; returns dict of DataFrames."""
    ids = [dict(kv.split("=", 1) for kv in ln.split())["id"] for ln in lines]
    dup = sorted({i for i in ids if ids.count(i) > 1})
    assert not dup, f"duplicate job ids (their outputs would merge): {dup[:5]}"
    out = WORK / name
    shutil.rmtree(out, ignore_errors=True)
    out.mkdir(parents=True)
    if not (WORK / "data").exists():
        (WORK / "data").symlink_to(ROOT / "data")
    jf = out / "jobs.txt"
    jf.write_text("\n".join(lines) + "\n")
    t0 = time.time()
    sh(f"{BIN} run --jobs {jf} --out {out}/res --concurrent {CONCURRENT} --chunk {CHUNK}{POOL_FLAG}{EXTRA_FLAGS}",
       cwd=WORK, quiet=True)
    res = {k: pd.read_csv(out / "res" / f"{k}_out.csv" if k == "jobs" else out / "res" / f"{k}.csv")
           for k in ("jobs", "epochs", "steps", "preds")}
    bad = res["jobs"][res["jobs"].status != "ok"]
    print(f"{len(lines)} models in {time.time() - t0:.1f} s" + (f"; NOT ok: {len(bad)}" if len(bad) else ""))
    return res


def curves(res, metric="val_mse", ax=None):
    """Validation curve per tune job (the dot marks its best epoch)."""
    ax = ax or plt.subplots(figsize=(9, 4))[1]
    best = res["jobs"].set_index("id").best_epoch
    for i, g in res["epochs"].dropna(subset=[metric]).groupby("id"):
        ax.plot(g.epoch, g[metric], lw=1, label=i)
        b = g[g.epoch == best.get(i, -1)]
        ax.plot(b.epoch, b[metric], "o", ms=4, color=ax.lines[-1].get_color())
    ax.set_xlabel("epoch"); ax.set_ylabel(metric); ax.set_yscale("log")
    if res["epochs"].id.nunique() <= 12:
        ax.legend(fontsize=7)
    return ax


def qlike(y, f, eps=1e-3):
    """QLIKE of the variance forecast f^2 against y^2 (the diurnal baseline cancels): r - log r - 1,
    r = y^2 / (f^2 + eps^2) -- the trainer's smooth floor (qlike_eps); y = 0 rows get y^2 = 1e-12."""
    r = np.maximum(np.asarray(y, float) ** 2, 1e-12) / (np.asarray(f, float) ** 2 + eps**2)
    return r - np.log(r) - 1


def forecasts(res, design):
    """Forecasts (kind fc or val) with the date, the target y and the per-row losses."""
    p = res["preds"].copy()
    p["date"] = DATES.date.to_numpy()[p.row.to_numpy()]
    p["y"] = Y[design][p.row.to_numpy()]
    p["sq_err"] = (p.f - p.y) ** 2
    p["qlike"] = qlike(p.y, p.f)
    return p


def select(tune, design, keys=("L", "H", "layers", "batch", "lr")):
    """The cluster rule (phase2_select.py): per configuration, the MSE of the seed-averaged validation forecast;
    the lowest wins. Returns (configuration, {seed: that seed's best_epoch}, every configuration's score)."""
    j = tune["jobs"]
    assert (j.status == "ok").all(), j[j.status != "ok"][["id", "status"]]
    keys = [k for k in keys if k in j]
    v = forecasts(tune, design)
    v = v[v.kind == "val"].merge(j[["id", *keys]], on="id")
    avg = v.groupby([*keys, "row", "y"], as_index=False).f.agg(f="mean", n="size")
    assert avg.n.nunique() == 1, "configurations with different seed counts"
    tab = (avg.assign(se=(avg.f - avg.y) ** 2).groupby(keys).se.agg(val_mse="mean", n_val="size")
           .reset_index().sort_values(["val_mse", *keys]))
    cfg = tab.iloc[0][keys].to_dict()
    cfg = {k: (float(x) if k == "lr" else int(x)) for k, x in cfg.items()}
    sub = j.merge(pd.DataFrame([cfg]), on=keys)
    return cfg, {int(s): int(e) for s, e in zip(sub.seed, sub.best_epoch)}, tab


def keep(name):
    """Copy an experiment's folder to Drive (MyDrive/puffer_forecast/manual/<name>)."""
    shutil.copytree(WORK / name, pathlib.Path(DRIVE_DIR) / "manual" / name, dirs_exist_ok=True)
    print("saved to", pathlib.Path(DRIVE_DIR) / "manual" / name)

In [ ]:
# ---- example: tune a small grid on the 2,000 sessions ending 2019-12-31 (Phase 1's window), 2 seeds
w0, w1 = window("2019-12-31")
D = "all_features"
grid = [dict(L=L, H=H, lr=lr) for L in (5, 20) for H in (16, 64) for lr in (0.01, 0.03)]
lines = [job(data=f"data/{D}", w0=w0, w1=w1, seed=s, **g) for g in grid for s in SEEDS]
tune = train(lines, "tune_example")
curves(tune)
cfg, best_epoch, tab = select(tune, D)      # the cluster's selection rule
print("selected", cfg, "| best epoch per seed", best_epoch)
tab.head(10)

In [ ]:
# ---- example: refit the selected configuration on the whole window, each seed at its own best epoch (the cluster
# rule), and forecast the next 10 sessions
lines = [job(data=f"data/{D}", w0=w0, w1=w1, mode="refit", epochs=best_epoch[s], fc_n=10, seed=s, **cfg)
         for s in SEEDS]
refit = train(lines, "refit_example")
fc = forecasts(refit, D)
avg = fc.groupby(["row", "date", "y"], as_index=False).f.mean()      # the seed average of f is the forecast
avg["qlike"] = qlike(avg.y, avg.f)
# fit-space QLIKE with the trainer's eps over these 10 sessions (all sessions, not just the paper's trade days)
print(cfg, "| mean QLIKE", round(avg.qlike.mean(), 4), "| MSE", round(((avg.f - avg.y)**2).mean(), 5))
avg

In [ ]:
# ---- your experiments: copy the two cells above and change the job lines; keep("<name>") saves a run to Drive

## Part B: the full walk-forward (set `RUN_WALK = True` in the settings cell)

Same steps as the cluster job (`cluster/puffer_forecast_phase2c_carc.sbatch`): for each block, tune the grid on the block's window → `phase2_select.py` picks the configuration → cold refits at every session (5 seeds). Each finished block is copied to Drive, so a disconnect loses at most one block; running again resumes (it refuses if the code or inputs changed). At the end one zip lands in `MyDrive/puffer_forecast/runs/`; score it on the laptop with `python experiments/puffer_forecast/phase2_score.py <extracted dir> <phase>_colab`.

In [ ]:
# ---- Part B
def to_drive(rel):
    src, dst = RUN / rel, DRIVE_RUN / rel
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)


def mark_done(rel, seconds):
    """Written on Drive only after the copy, so a half-copied block is never taken as finished."""
    stamp = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    (DRIVE_RUN / rel / "DONE").write_text(f"{GPU} | {seconds:.0f} s | {stamp}\n")


def walk():
    global RUN, DRIVE_RUN
    RUN = pathlib.Path("/content/runs") / RUN_NAME
    DRIVE_RUN = pathlib.Path(DRIVE_DIR) / "runs" / RUN_NAME
    RUN.mkdir(parents=True, exist_ok=True)
    DRIVE_RUN.mkdir(parents=True, exist_ok=True)
    if not (RUN / "data").exists():
        (RUN / "data").symlink_to(ROOT / "data")   # the job files read data=data/<design> from the run dir
    code_md5 = sh(f"md5sum native/* phase2_select.py inputs/{PHASE}/* data/*/*", cwd=ROOT)  # code, inputs AND data
    prev = DRIVE_RUN / "code_md5.txt"
    if prev.exists() and prev.read_text() != code_md5:
        raise RuntimeError(f"{DRIVE_RUN} holds blocks from different code or inputs: set a new RUN_NAME")
    (RUN / "code_md5.txt").write_text(code_md5)
    env = sh(f"hostname; date; nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader; "
             f"nvcc --version | tail -1; gcc --version | head -1; git -C {PUF} log -1 --format='%H %ci'")
    (RUN / "env.txt").write_text(env + "bundle " + json.dumps(INFO) + f"\nflags {POOL_FLAG}\n")
    shutil.copy2(BUILD / "build.log", RUN / "build.log")
    for f in ("code_md5.txt", "env.txt", "build.log"):
        to_drive(f)
    inp = ROOT / "inputs" / PHASE
    mc = inp / "maskcheck.txt"
    if mc.exists() and not (DRIVE_RUN / "maskcheck" / "DONE").exists():
        t0 = time.time()
        sh(f"{BIN} gradcheck --jobs {mc} --out {RUN}/maskcheck > {RUN}/maskcheck.log 2>&1 "
           f"|| (cat {RUN}/maskcheck.log; exit 1)", cwd=RUN)
        to_drive("maskcheck")
        to_drive("maskcheck.log")
        mark_done("maskcheck", time.time() - t0)
    ks = sorted(int(p.stem.removeprefix("tune_block")) for p in inp.glob("tune_block*.txt"))
    for k in ks:
        b = f"block{k}"
        if (DRIVE_RUN / b / "DONE").exists():
            print(b, "is on Drive already:", (DRIVE_RUN / b / "DONE").read_text().strip())
            continue
        shutil.rmtree(RUN / b, ignore_errors=True)
        (RUN / b).mkdir()
        t0 = time.time()
        print(f"BLOCK {k} tune {datetime.datetime.now()}")
        sh(f"{BIN} run --jobs {inp}/tune_block{k}.txt --out {RUN}/{b}/tune --concurrent {CONCURRENT} --chunk {CHUNK}{POOL_FLAG}", cwd=RUN)
        sh(f"python3 {ROOT}/phase2_select.py --blocks {inp}/blocks.json --k {k} --tune-out {RUN}/{b}/tune "
           f"--data-root {ROOT}/data --out {RUN}/{b}/jobs_refit.txt --selection {RUN}/{b}/selection.json", cwd=RUN)
        print(f"BLOCK {k} refit {datetime.datetime.now()}")
        sh(f"{BIN} run --jobs {RUN}/{b}/jobs_refit.txt --out {RUN}/{b}/refit --concurrent {CONCURRENT} --chunk {CHUNK}{POOL_FLAG}", cwd=RUN)
        to_drive(b)
        mark_done(b, time.time() - t0)
        print(f"BLOCK {k} done in {time.time() - t0:.0f} s, copied to Drive")
    (DRIVE_RUN / "PHASE2_DONE").write_text(datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds") + "\n")
    zpath = DRIVE_RUN.parent / f"{RUN_NAME}.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(DRIVE_RUN.rglob("*")):
            if p.is_file():
                z.write(p, p.relative_to(DRIVE_RUN).as_posix())
    print(zpath, f"{zpath.stat().st_size / 1e6:.1f} MB, md5", hashlib.md5(zpath.read_bytes()).hexdigest())


if RUN_WALK:
    walk()
else:
    print("Part B skipped (RUN_WALK = False)")